In [1]:
# Cell 1: Imports and setup
import numpy as np
from ase.io import read, write
from ase.build import make_supercell
from ase.optimize import BFGS
from mace.calculators import mace_mp
from collections import Counter
import json
import os

print("✓ Imports complete")

C:\Users\Kai Savage\anaconda3\Lib\site-packages\e3nn\o3\_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
✓ Imports complete


In [3]:
# Cell 2: Load MACE calculator
calc = mace_mp(
    model='mace-mh-1.model',
    default_dtype='float64',
    device='cpu',
    head='omat_pbe'
)

print("✓ MACE calculator loaded")

Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


C:\Users\Kai Savage\anaconda3\Lib\site-packages\mace\calculators\mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


✓ MACE calculator loaded


In [4]:
# Cell 3: Load O2 and H2 reference energies
with open('molecular_references.json', 'r') as f:
    mol_refs = json.load(f)

E_O2 = mol_refs['O2']['E_total']
E_H2 = mol_refs['H2']['E_total']

E_ref_O = 0.5 * E_O2
E_ref_H = 0.5 * E_H2
E_ref_OH = E_ref_O + E_ref_H

print(f"O2 reference: {E_O2:.6f} eV → ½O2 = {E_ref_O:.6f} eV")
print(f"H2 reference: {E_H2:.6f} eV → ½H2 = {E_ref_H:.6f} eV")
print(f"OH reference: {E_ref_OH:.6f} eV")

O2 reference: -9.896588 eV → ½O2 = -4.948294 eV
H2 reference: -6.766273 eV → ½H2 = -3.383137 eV
OH reference: -8.331431 eV


In [6]:
# Cell 4: Build 2×2 supercell from 1×1 clean slab

# Load 1×1 clean slab
slab_1x1 = read('mgzn_1.xyz', index=-1)  # -1 = last frame

# Deduce cell from coordinates
coords = slab_1x1.get_positions()
x_min, x_max = coords[:,0].min(), coords[:,0].max()
y_min, y_max = coords[:,1].min(), coords[:,1].max()
z_min, z_max = coords[:,2].min(), coords[:,2].max()

a = x_max - x_min + 0.5
b = y_max - y_min + 0.5
c = z_max - z_min + 20.0

# Set cell and PBC
slab_1x1.set_cell([[a, 0, 0], [0, b, 0], [0, 0, c]])
slab_1x1.set_pbc([True, True, False])

# Shift to positive coordinates
slab_1x1.positions += np.array([a/2, b/2, c/2])

print(f"1×1 slab: {len(slab_1x1)} atoms")
print(f"Formula: {slab_1x1.get_chemical_formula()}")

# Build 2×2 supercell
P = [[2, 0, 0], [0, 2, 0], [0, 0, 1]]
clean_2x2 = make_supercell(slab_1x1, P)

print(f"\n2×2 supercell: {len(clean_2x2)} atoms")
print(f"Formula: {clean_2x2.get_chemical_formula()}")

# Save initial structure
write('clean_slab_2x2_initial.xyz', clean_2x2)
print("✓ Saved to: clean_slab_2x2_initial.xyz")

1×1 slab: 24 atoms
Formula: Mg8Zn16

2×2 supercell: 96 atoms
Formula: Mg32Zn64
✓ Saved to: clean_slab_2x2_initial.xyz


In [7]:
# Cell 5: Relax 2×2 clean slab with MACE

clean_2x2.calc = calc

# Initial energy
E_initial = clean_2x2.get_potential_energy()
print(f"Initial energy: {E_initial:.6f} eV")

# Optimize
opt = BFGS(clean_2x2, trajectory='clean_slab_2x2.traj')
opt.run(fmax=0.05)

# Final energy
E_clean = clean_2x2.get_potential_energy()
print(f"Final energy: {E_clean:.6f} eV")
print(f"Energy change: {E_clean - E_initial:.6f} eV")
print(f"Steps: {opt.get_number_of_steps()}")

# Save relaxed structure
write('clean_slab_2x2_relaxed.xyz', clean_2x2)
print(f"\n✓ E_clean = {E_clean:.6f} eV")

Initial energy: -106.274700 eV
      Step     Time          Energy          fmax
BFGS:    0 18:04:12     -106.274700        0.659817
BFGS:    1 18:04:23     -106.461536        0.612290
BFGS:    2 18:04:30     -108.283212        0.444498
BFGS:    3 18:04:36     -108.356238        0.448723
BFGS:    4 18:04:41     -108.695654        0.321648
BFGS:    5 18:04:47     -108.913852        0.238260
BFGS:    6 18:04:53     -109.011479        0.254182
BFGS:    7 18:04:59     -109.042127        0.253873
BFGS:    8 18:05:04     -109.117602        0.237310
BFGS:    9 18:05:10     -109.222220        0.198559
BFGS:   10 18:05:16     -109.350705        0.214712
BFGS:   11 18:05:22     -109.424230        0.164704
BFGS:   12 18:05:27     -109.448350        0.160205
BFGS:   13 18:05:33     -109.463512        0.150363
BFGS:   14 18:05:39     -109.488317        0.095895
BFGS:   15 18:05:44     -109.506409        0.085984
BFGS:   16 18:05:50     -109.514394        0.064130
BFGS:   17 18:05:56     -109.516926

In [13]:
# Cell 6: Function to add cell and PBC to XYZ files (with debugging)

def convert_xyz_add_cell(input_file, output_file):
    """
    Read XYZ, extract last frame, add cell and PBC, save.
    """
    try:
        # Read all frames
        with open(input_file, 'r') as f:
            lines = f.readlines()
        
        # Find frame starts
        frame_starts = []
        for i, line in enumerate(lines):
            try:
                natoms = int(line.strip())
                frame_starts.append((i, natoms))
            except:
                continue
        
        if not frame_starts:
            raise ValueError(f"No frames found in {input_file}")
        
        # Get last frame
        last_start, natoms = frame_starts[-1]
        last_frame_lines = lines[last_start : last_start + natoms + 2]
        
        # Parse atoms
        elements = []
        coords = []
        for line in last_frame_lines[2:]:
            parts = line.split()
            if len(parts) >= 4:
                elem = parts[0].strip()
                # Normalize element symbols: ZN -> Zn, MG -> Mg
                if elem.upper() == 'ZN':
                    elem = 'Zn'
                elif elem.upper() == 'MG':
                    elem = 'Mg'
                elif elem.upper() == 'O':
                    elem = 'O'
                elif elem.upper() == 'H':
                    elem = 'H'
                
                elements.append(elem)
                coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
        
        coords = np.array(coords)
        
        # Deduce cell
        x_min, x_max = coords[:,0].min(), coords[:,0].max()
        y_min, y_max = coords[:,1].min(), coords[:,1].max()
        z_min, z_max = coords[:,2].min(), coords[:,2].max()
        
        a = x_max - x_min + 0.5
        b = y_max - y_min + 0.5
        c = z_max - z_min + 20.0
        
        # Create atoms object
        from ase import Atoms
        atoms = Atoms(
            symbols=elements,
            positions=coords + np.array([a/2, b/2, c/2]),
            cell=[[a, 0, 0], [0, b, 0], [0, 0, c]],
            pbc=[True, True, False]
        )
        
        # Save
        write(output_file, atoms)
        return atoms
        
    except Exception as e:
        print(f"    Error details: {str(e)}")
        raise

print("✓ Function defined")

✓ Function defined


In [14]:
# Cell 7: Convert all adsorbate structures (add cell + PBC)

# Define all adsorbate files with correct folder structure
adsorbate_files = {
    'O': {
        'Mg': 'O_Mg.xyz',
        'Zn': 'O_Zn.xyz',
        'inbetween-Mg': 'O_inbetween_Mg.xyz'
    },
    'H': {
        'Mg': 'H_Mg.xyz',
        'Zn': 'H_Zn.xyz',
        'inbetween-Mg': 'H_inbetween_Mg.xyz'
    },
    'OH': {
        'Mg': 'OH_Mg.xyz',
        'inbetween-Mg': 'OH_inbetween_Mg.xyz'  # No Zn for OH
    }
}

converted_files = []

for ads_type, sites in adsorbate_files.items():
    print(f"\n{ads_type} adsorbates:")
    for site, fname in sites.items():
        # Correct path:O/Mg/O_Mg.xyz
        input_path = f'{ads_type}/{site}/{fname}'
        output_path = f'{fname.replace(".xyz", "_converted.xyz")}'
        
        try:
            atoms = convert_xyz_add_cell(input_path, output_path)
            converted_files.append(output_path)
            print(f"  ✓ {site:15s}: {fname} → {output_path} ({len(atoms)} atoms)")
        except Exception as e:
            print(f"  ✗ {site:15s}: {fname} - Error: {e}")

print(f"\n✓ Converted {len(converted_files)} files")



O adsorbates:
  ✓ Mg             : O_Mg.xyz → O_Mg_converted.xyz (112 atoms)
  ✓ Zn             : O_Zn.xyz → O_Zn_converted.xyz (104 atoms)
  ✓ inbetween-Mg   : O_inbetween_Mg.xyz → O_inbetween_Mg_converted.xyz (104 atoms)

H adsorbates:
  ✓ Mg             : H_Mg.xyz → H_Mg_converted.xyz (112 atoms)
  ✓ Zn             : H_Zn.xyz → H_Zn_converted.xyz (104 atoms)
  ✓ inbetween-Mg   : H_inbetween_Mg.xyz → H_inbetween_Mg_converted.xyz (104 atoms)

OH adsorbates:
  ✓ Mg             : OH_Mg.xyz → OH_Mg_converted.xyz (128 atoms)
  ✓ inbetween-Mg   : OH_inbetween_Mg.xyz → OH_inbetween_Mg_converted.xyz (112 atoms)

✓ Converted 8 files


In [15]:
# Cell 8: Relax all adsorbate structures with MACE

relaxed_energies = {}

for fname in converted_files:
    name = fname.replace('_converted.xyz', '')
    
    print(f"\n{'='*60}")
    print(f"Relaxing: {name}")
    print(f"{'='*60}")
    
    # Load structure
    atoms = read(fname)
    atoms.calc = calc
    
    # Count composition
    symbols = atoms.get_chemical_symbols()
    counts = Counter([s.upper() for s in symbols])
    print(f"Composition: {dict(counts)}")
    
    # Initial energy
    E_initial = atoms.get_potential_energy()
    print(f"Initial: {E_initial:.6f} eV")
    
    # Relax
    opt = BFGS(atoms, trajectory=f'{name}_relax.traj')
    opt.run(fmax=0.05)
    
    # Final energy
    E_final = atoms.get_potential_energy()
    print(f"Final: {E_final:.6f} eV")
    print(f"ΔE: {E_final - E_initial:.6f} eV")
    print(f"Steps: {opt.get_number_of_steps()}")
    
    # Save relaxed structure
    write(f'{name}_relaxed.xyz', atoms)
    
    # Store results
    relaxed_energies[name] = {
        'E_total': E_final,
        'n_atoms': len(atoms),
        'composition': dict(counts)
    }

print(f"\n✓ Relaxed {len(relaxed_energies)} structures")


Relaxing: O_Mg
Composition: {'ZN': 64, 'MG': 32, 'O': 16}
Initial: -204.794071 eV
      Step     Time          Energy          fmax
BFGS:    0 18:15:19     -204.794071        0.746071
BFGS:    1 18:15:29     -204.942788        0.701170
BFGS:    2 18:15:36     -206.375628        1.069046
BFGS:    3 18:15:44     -206.530195        0.597101
BFGS:    4 18:15:52     -206.831975        0.580990
BFGS:    5 18:15:59     -206.972441        0.794671
BFGS:    6 18:16:06     -207.145318        0.770464
BFGS:    7 18:16:12     -207.301476        0.518763
BFGS:    8 18:16:18     -207.442837        0.572233
BFGS:    9 18:16:24     -207.577909        0.858587
BFGS:   10 18:16:31     -207.741519        0.798208
BFGS:   11 18:16:37     -207.865696        0.550809
BFGS:   12 18:16:43     -208.019746        0.624129
BFGS:   13 18:16:53     -208.289750        1.128331
BFGS:   14 18:17:01     -208.562668        1.255031
BFGS:   15 18:17:09     -208.802011        0.989874
BFGS:   16 18:17:15     -209.123368

In [16]:
# Cell 9: Calculate binding energies

binding_energies = {}

for name, data in relaxed_energies.items():
    # Determine adsorbate type and count
    comp = data['composition']
    n_O = comp.get('O', 0)
    n_H = comp.get('H', 0)
    
    # Figure out adsorbate type
    if n_O > 0 and n_H == 0:
        ads_type = 'O'
        n_ads = n_O
        E_ref = E_ref_O
    elif n_H > 0 and n_O == 0:
        ads_type = 'H'
        n_ads = n_H
        E_ref = E_ref_H
    elif n_O > 0 and n_H > 0:
        ads_type = 'OH'
        n_ads = n_O  # OH count = O count
        E_ref = E_ref_OH
    else:
        continue
    
    # Calculate binding energy per adsorbate
    E_bind = (data['E_total'] - E_clean - n_ads * E_ref) / n_ads
    
    binding_energies[name] = {
        'adsorbate': ads_type,
        'n_adsorbates': n_ads,
        'E_total': data['E_total'],
        'E_bind_per_ads': E_bind
    }
    
    print(f"{name:20s}: {E_bind:+.3f} eV/ads ({n_ads} {ads_type})")

print(f"\n✓ Calculated {len(binding_energies)} binding energies")

O_Mg                : -2.132 eV/ads (16 O)
O_Zn                : -3.707 eV/ads (8 O)
O_inbetween_Mg      : -3.791 eV/ads (8 O)
H_Mg                : -0.026 eV/ads (16 H)
H_Zn                : +1.704 eV/ads (8 H)
H_inbetween_Mg      : -0.203 eV/ads (8 H)
OH_Mg               : -3.591 eV/ads (16 OH)
OH_inbetween_Mg     : -4.067 eV/ads (8 OH)

✓ Calculated 8 binding energies


In [18]:
# Cell 9: Calculate binding energies (CORRECTED with proper references)

# Extract energies from relaxation run
energies_data = {
    'O_Mg': {'E_single_point': -204.794071, 'E_relaxed': -222.811051, 'n_O': 16, 'n_H': 0},
    'O_Zn': {'E_single_point': -153.056255, 'E_relaxed': -178.762430, 'n_O': 8, 'n_H': 0},
    'O_inbetween_Mg': {'E_single_point': -171.585335, 'E_relaxed': -179.435035, 'n_O': 8, 'n_H': 0},
    'H_Mg': {'E_single_point': -152.924800, 'E_relaxed': -164.066568, 'n_O': 0, 'n_H': 16},
    # 'H_Zn': EXCLUDED - corrupted geometry in DFT file
    'H_inbetween_Mg': {'E_single_point': -135.237539, 'E_relaxed': -138.213645, 'n_O': 0, 'n_H': 8},
    'OH_Mg': {'E_single_point': -287.094925, 'E_relaxed': -300.281814, 'n_O': 16, 'n_H': 16},
    'OH_inbetween_Mg': {'E_single_point': -205.715605, 'E_relaxed': -208.706175, 'n_O': 8, 'n_H': 8}
}

# Reference energies
E_clean_SP = -106.274700      # MACE single-point on DFT clean slab geometry
E_clean_relax = -109.522219   # MACE energy on relaxed clean slab geometry

binding_energies_single_point = {}
binding_energies_relaxed = {}

print("="*80)
print("BINDING ENERGIES (eV/adsorbate)")
print("="*80)
print(f"\nUsing:")
print(f"  E_clean (single-point) = {E_clean_SP:.6f} eV  (MACE on DFT geometry)")
print(f"  E_clean (relaxed)      = {E_clean_relax:.6f} eV  (MACE on MACE geometry)")
print(f"  E_ref_O  = {E_ref_O:.6f} eV")
print(f"  E_ref_H  = {E_ref_H:.6f} eV")
print(f"  E_ref_OH = {E_ref_OH:.6f} eV")

print(f"\n{'Structure':<20s} {'Single-Point':>12s} {'Relaxed':>12s} {'Δ(Relax)':>12s}")
print("-"*80)

for name, data in energies_data.items():
    n_O = data['n_O']
    n_H = data['n_H']
    
    # Determine adsorbate type
    if n_O > 0 and n_H == 0:
        ads_type = 'O'
        n_ads = n_O
        E_ref = E_ref_O
    elif n_H > 0 and n_O == 0:
        ads_type = 'H'
        n_ads = n_H
        E_ref = E_ref_H
    else:  # OH
        ads_type = 'OH'
        n_ads = n_O
        E_ref = E_ref_OH
    
    # Single-point binding energy (MACE on DFT geometry)
    E_bind_sp = (data['E_single_point'] - E_clean_SP - n_ads * E_ref) / n_ads
    
    # Relaxed binding energy (MACE on MACE geometry)
    E_bind_relax = (data['E_relaxed'] - E_clean_relax - n_ads * E_ref) / n_ads
    
    diff = E_bind_relax - E_bind_sp
    
    binding_energies_single_point[name] = {
        'adsorbate': ads_type,
        'n_adsorbates': n_ads,
        'E_total': data['E_single_point'],
        'E_bind_per_ads': E_bind_sp
    }
    
    binding_energies_relaxed[name] = {
        'adsorbate': ads_type,
        'n_adsorbates': n_ads,
        'E_total': data['E_relaxed'],
        'E_bind_per_ads': E_bind_relax
    }
    
    print(f"{name:<20s} {E_bind_sp:>+12.3f} {E_bind_relax:>+12.3f} {diff:>+12.3f}")

print("\n" + "="*80)
print(f"✓ Calculated {len(binding_energies_single_point)} binding energies")
print("✗ H_Zn excluded (corrupted geometry in DFT file)")
print("="*80)


BINDING ENERGIES (eV/adsorbate)

Using:
  E_clean (single-point) = -106.274700 eV  (MACE on DFT geometry)
  E_clean (relaxed)      = -109.522219 eV  (MACE on MACE geometry)
  E_ref_O  = -4.948294 eV
  E_ref_H  = -3.383137 eV
  E_ref_OH = -8.331431 eV

Structure            Single-Point      Relaxed     Δ(Relax)
--------------------------------------------------------------------------------
O_Mg                       -1.209       -2.132       -0.923
O_Zn                       -0.899       -3.707       -2.807
O_inbetween_Mg             -3.216       -3.791       -0.575
H_Mg                       +0.468       -0.026       -0.493
H_inbetween_Mg             -0.237       -0.203       +0.034
OH_Mg                      -2.970       -3.591       -0.621
OH_inbetween_Mg            -4.099       -4.067       +0.032

✓ Calculated 7 binding energies
✗ H_Zn excluded (corrupted geometry in DFT file)


In [33]:
# Cell: Relax properly converted H_Zn

from ase.io import read, write
from ase.optimize import BFGS

atoms = read('H_Zn_CORRECTED_FINAL.xyz')
atoms.calc = calc

print("Relaxing properly converted H_Zn...")

# Initial energy
E_initial = atoms.get_potential_energy()
print(f"Initial: {E_initial:.6f} eV")

# Relax
opt = BFGS(atoms, trajectory='H_Zn_proper_relax.traj')
opt.run(fmax=0.05)

# Final energy
E_final = atoms.get_potential_energy()
print(f"Final:   {E_final:.6f} eV")
print(f"Steps:   {opt.get_number_of_steps()}")

# Save
write('H_Zn_relaxed_PROPER.xyz', atoms)

print(f"\nUpdate Cell 9 with:")
print(f"'H_Zn': {{'E_single_point': {E_initial:.6f}, 'E_relaxed': {E_final:.6f}, 'n_O': 0, 'n_H': 8}},")

Relaxing properly converted H_Zn...
Initial: -119.462819 eV
      Step     Time          Energy          fmax
BFGS:    0 11:50:37     -119.462819        0.654516
BFGS:    1 11:50:47     -119.573697        0.616916
BFGS:    2 11:50:54     -120.841884        0.612405
BFGS:    3 11:51:00     -120.941860        0.526014
BFGS:    4 11:51:06     -121.394975        0.667007
BFGS:    5 11:51:12     -121.511331        0.689457
BFGS:    6 11:51:18     -121.751656        0.661233
BFGS:    7 11:51:23     -121.840895        0.411597
BFGS:    8 11:51:29     -121.929333        0.398392
BFGS:    9 11:51:35     -121.987883        0.569348
BFGS:   10 11:51:40     -122.109866        0.803627
BFGS:   11 11:51:46     -122.188954        0.529981
BFGS:   12 11:51:52     -122.228871        0.275443
BFGS:   13 11:51:57     -122.264268        0.297153
BFGS:   14 11:52:03     -122.312826        0.496233
BFGS:   15 11:52:09     -122.361482        0.619273
BFGS:   16 11:52:16     -122.406535        0.507146
BFGS: 

In [35]:
# Cell: H_Zn binding energies - simple calculation

# Known values
E_clean_SP = -106.274700      # eV (clean slab, single-point)
E_clean_relax = -109.522219   # eV (clean slab, relaxed)
E_ref_H = -3.383500           # eV (½H₂ reference)

E_H_Zn_SP = -119.462819       # eV (H_Zn single-point)
E_H_Zn_relax = -122.663566    # eV (H_Zn relaxed)
n_H = 8

# Binding energies
E_bind_SP = (E_H_Zn_SP - E_clean_SP - n_H * E_ref_H) / n_H
E_bind_relax = (E_H_Zn_relax - E_clean_relax - n_H * E_ref_H) / n_H

print("H_Zn Binding Energies:")
print(f"  Single-point: {E_bind_SP:+.3f} eV/H")
print(f"  Relaxed:      {E_bind_relax:+.3f} eV/H")

H_Zn Binding Energies:
  Single-point: +1.735 eV/H
  Relaxed:      +1.741 eV/H
